### **CREATE TABLE AS SELECT (CTAS)**

Creates a Delta table by default from files in cloud object storage

The `read_files()` function reads files under a provided location and returns the data in tabular form.

- Supports reading file formats like JSON, CSV, XML, TEXT, BINARYFILE, PARQUET, AVRO, and ORC file formats.

- Can detect the **file format automatically and infer a unified schema** across all files.

- Specify **specific file format options** to read in the data based on the source file format.

- Can be used in **streaming tables** to **incrementally** ingest files into Delta Lake using Auto Loader.

In [0]:
%sql
CREATE TABLE new_table
AS SELECT
  *
FROM read_files(
  'path/to/files',
  format => 'file_type',
  -- other_format_specific_options
);

### **COPY INTO**

- Is a **retriable and idempotent operation** and will skip files that have already been loaded (**incremental**)

- Supports various common files types like parquet, JSON, XML, etc

- **FROM** specifies the path of the cloud storage location continuously adding files

- **FORMAT_OPTIONS()** control how the source files are parsed and interpreted. The available options depends on the file format

- **COPY_OPTIONS()** controls the behavior of the COPY INTO operation itself, such as schema evolution (**mergeSchema**) and idempotency (**force**)

**1. Create an empty table to copy data into**

- You can create an empty table without a schema

- You also can explicitly create the table with a schema

In [0]:
%sql
CREATE TABLE new_table;

COPY INTO new_table
FROM
  'path/to/files'
FILEFORMAT = file_type
FORMAT_OPTIONS(options)
COPY_OPTIONS(options);

### **Auto Loader**

- Incrementally and efficiently processes new data files (in batch or streaming) as they arrive in cloud storage without any additional setup
- Auto Loader has support for both Python and SQL (leveraging Declarative Pipelines)
- You can use Auto Loader to process billions of files
- Auto Loader is built upon Spark Structured Streaming
- A deep dive into Auto Loader is out of scope for this course, please refer to these links and courses for more in depth information resources:

In [0]:
(spark
.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_location)
    .load("/Volumes/catalog/schema/files")
.writeStream
    .option("checkpointLocation", checkpoint_location)
    .trigger(processingTime="5 seconds")
    .toTable("catalog.database.table")
)

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE
  catalog.schema.table_name
SCHEDULE EVERY
  1 HOUR
AS SELECT
  *
FROM STREAM read_files(
  'dir_path',
  format => 'file_type'
);

| FEATURE                | CREATE TABLE AS (CTAS) + spark.read | COPY INTO                     | Auto Loader                                                                 |
|------------------------|-------------------------------------|-------------------------------|-----------------------------------------------------------------------------|
| **Ingestion Type**         | Batch                               | Incremental Batch             | Incremental (Batch or Streaming)                                            |
| **Use Cases**              | Best for smaller datasets           | Ideal for thousands of files  | Scale to millions+ of files per hour, backfills with billions of files     |
| **Syntax/Interface**       | ● Python (`spark.read`)               | SQL                           | ● Python (`spark.readStream`)                                                 |
|                        | ● SQL (CTAS)                        |                               | ● SQL with Declarative Pipelines (CREATE OR REFRESH STREAMING TABLES)       |
|                        |                                     |                               | ○ Use streaming tables in Databricks SQL                                    |
| **Idempotency**            | No                                  | Yes                           | Yes                                                                         |
| **Schema Evolution**       | Manual or inferred during read      | Supported with options        | Auto Loader automatically detects and evolves schemas. It supports loading data without predefined schemas and handles new columns as they appear. |
| **Latency**                | High                                | Moderate (scheduled)          | Low or high depending configuration                                        |
| Ease of Use            | Simple                              | Simple and SQL-based          | Intermediate to advanced depending on the implementation (Python or SQL, incremental batch or streaming) |
| **Summary**                | Best for one time, ad hoc ingestion. Can be scheduled to always read and process all data. | Simple and repeatable for incremental file ingestion. Great for scheduled jobs or pipelines. | Best for near real-time streaming or incremental ingestion, with high automation and scalability. |

### **Appending Metadata Columns on Ingest**

In data ingestion from cloud storage, metadata columns like source file name and modification time can be appended using the `_metadata` column.

- `_metadata.file_modification_time`: Provides the last modification timestamp of the input file.

- `_metadata.file_name`: Returns the name of the source file for each row.

In [0]:
%sql
SELECT
  _metadata.file_modification_time AS file_modification_time,
  _metadata.file_name AS file_name,
  current_timestamp() AS ingestion_time,
FROM
  read_files(
    'dir_path',
    format => 'file_type'
  );

In [0]:
from pyspark.sql.functions import col, from_unixtime, current_timestamp
from pyspark.sql.types import DateType

df = spark.read.format("csv").load("some_csv/path/file.csv")

df_with_metadata = (
    df.withColumn("first_touch_date", from_unixtime(col("user_first_touch_timestamp") / 1_000_000).cast(DateType()))
      .withColumn("file_modification_time", col("_metadata.file_modification_time"))
      .withColumn("source_file", col("_metadata.file_name"))
      .withColumn("ingestion_time", current_timestamp())
)

### **Rescued Data**

During data ingestion there are times when the input data doesn't match with the schema in your table. Ingestion techniques like `read_files()`, `spark.read` or Auto Loader provide a rescued data column during ingestion. The rescued data column **ensures that columns that don't match with the schema are rescued instead of being dropped.**

![](https://cdn5.dcbstatic.com/files/d/a/databricks_docebosaas_com/1767819600/VyoFcUu7LxPchoEE_q4uyQ/authoring/674/674_full_slide3_1.jpg)

In [0]:
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  # Aquí defines el nombre de la columna que guardará los errores
  .option("rescuedDataColumn", "_rescued_data") 
  .option("cloudFiles.schemaLocation", path_schema)
  .load(path_data))

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE my_table
AS
SELECT
  *,
  _rescued_data
FROM read_files(
  's3://your-bucket/path/',
  format => 'json',
  rescuedDataColumn => '_rescued_data'
);